# MR_POKER - SFT Fine-tune Qwen2.5-1.5B no PokerBench

**Modelo:** Qwen/Qwen2.5-1.5B-Instruct + LoRA (r=16, alpha=32)
**Dataset:** felipesp1983/pokerbench-sft-chat
**Output:** felipesp1983/poker-solver-qwen-1.5b-sft

Requer: Colab Pro com GPU (A100/V100/T4)

In [ ]:
# Cell 1: Instalar dependencias
!pip install -q trl>=0.12.0 peft>=0.7.0 datasets>=3.0.0 accelerate>=0.30.0 bitsandbytes>=0.43.0 huggingface_hub

In [ ]:
# Cell 2: Login no HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 3: Verificar GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {gpu_mem:.1f} GB")
    print(f"BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Cell 4: Carregar dataset e treinar
import torch
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Config
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "felipesp1983/pokerbench-sft-chat"
OUTPUT_MODEL = "felipesp1983/poker-solver-qwen-1.5b-sft"

# Carregar dataset
print(f"Carregando dataset: {DATASET_ID}")
train_ds = load_dataset(DATASET_ID, split="train")
eval_ds = load_dataset(DATASET_ID, split="test")
print(f"  train: {len(train_ds)} rows, test: {len(eval_ds)} rows")

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# Detectar GPU
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0

# Ajustar batch size pela GPU
if gpu_mem >= 40:  # A100
    batch_size, grad_accum = 8, 4
elif gpu_mem >= 16:  # V100/T4
    batch_size, grad_accum = 4, 8
else:
    batch_size, grad_accum = 2, 16

print(f"Config: batch={batch_size}, accum={grad_accum}, bf16={use_bf16}, fp16={use_fp16}")

# SFT config
training_args = SFTConfig(
    output_dir="./poker-solver-sft",
    num_train_epochs=1,
    learning_rate=2e-4,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    max_length=512,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=500,
    push_to_hub=True,
    hub_model_id=OUTPUT_MODEL,
    hub_strategy="every_save",
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
)

# Treinar
print(f"Carregando modelo: {BASE_MODEL}")
trainer = SFTTrainer(
    model=BASE_MODEL,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=lora_config,
    args=training_args,
)

steps = len(train_ds) // (batch_size * grad_accum)
print(f"Iniciando treino: {len(train_ds)} rows, ~{steps} steps, 1 epoch")
trainer.train()

In [ ]:
# Cell 5: Push modelo final para o Hub
print("Enviando modelo final para o HuggingFace Hub...")
trainer.push_to_hub()
print(f"PRONTO! Modelo em: https://huggingface.co/{OUTPUT_MODEL}")